# Can we predict next-month churn?
Data: monthly account snapshots (`data.csv`), target `churn_next_month`. Plan:
1. Inspect the data and check every column for leakage.
2. Build features that only use information available at the snapshot date.
3. Validate **out-of-time** (train on earlier months, test on later ones); the same account appears in many rows, so random row splits would be optimistic.
4. Compare against simple baselines.

In [ ]:
import pandas as pd, numpy as np, matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, average_precision_score
pd.set_option('display.width', 140)
DATA = '/private/tmp/waldean-agent-baseline/runs-v2/subscriptions-claude-default-1/'
df = pd.read_csv(DATA + 'data.csv')
print(df.shape, df.account_ref.nunique(), 'accounts')
print(pd.read_csv(DATA + 'columns.csv').to_string(index=False))
df.head()

In [ ]:
print(df.isna().sum().sum(), 'missing;', df.duplicated(['account_ref', 'snapshot_month']).sum(), 'duplicate account-months')
print('churn rate:', df.churn_next_month.mean().round(3))
df.describe().T

## 1. Leakage check: `lifetime_months`
Its description says it comes from a CRM export dated **2026-01-05**, after every snapshot. It records the account's total active life, which is only known once the account has ended. If it is leaky, `lifetime_months - tenure_months` (months remaining) should reveal churn.

In [ ]:
df['remaining'] = df.lifetime_months - df.tenure_months
print(df.groupby('churn_next_month').remaining.describe())
print('churn rows with remaining==0:', (df[df.churn_next_month == 1].remaining == 0).mean())
print('non-churn rows with remaining==0:', (df[df.churn_next_month == 0].remaining == 0).mean())
print('AUC of -remaining alone:', round(roc_auc_score(df.churn_next_month, -df.remaining), 3))

**Confirmed leakage.** Every churned row has `remaining == 0` and no non-churned row does, so the column encodes the outcome. It would not be known for a live account at prediction time, so it is **excluded** from all models. I show what it would score further down, for contrast only.

## 2. Churn over time

In [ ]:
m = df.groupby('snapshot_month').churn_next_month.agg(['mean', 'size'])
ax = m['mean'].plot(marker='o', figsize=(10, 3.5), title='Churn rate by snapshot month')
ax.set_ylabel('churn rate'); plt.xticks(rotation=60); plt.show()
m.T.round(3)

The churn rate is far from stable: ~40-55% for Nov 2024 - Jan 2025 versus ~3-11% in most other months (early 2024 is also high, on a small young base). So a random split would mix regimes, and one overall base rate is misleading. I split by time. With under two years of data I can't tell whether the winter spike is seasonal (will recur) or a one-off event.

In [ ]:
g = df.assign(tenure_bin=pd.cut(df.tenure_months, [0, 1, 2, 3, 6, 12, 24]),
              usage_bin=pd.qcut(df.usage_hours, 5, duplicates='drop'))
fig, ax = plt.subplots(1, 4, figsize=(16, 3.2))
for a, c in zip(ax, ['tenure_bin', 'usage_bin', 'support_tickets', 'plan']):
    g.groupby(c, observed=True).churn_next_month.mean().plot.bar(ax=a, title=c)
    a.set_xlabel('')
plt.tight_layout(); plt.show()

## 3. Features
Only information available at snapshot time:
- the snapshot's own columns plus fee per seat;
- the account's **previous** snapshots only (last month's usage, change, 3-month mean usage, ticket trend).

Calendar month-of-year is evaluated as a separate optional feature, since we can't verify the time pattern repeats.

In [ ]:
df = df.sort_values(['account_ref', 'snapshot_month']).reset_index(drop=True)
df['fee_per_seat'] = df.monthly_fee / df.seats
g = df.groupby('account_ref')
df['usage_prev'] = g.usage_hours.shift(1)
df['usage_chg'] = df.usage_hours - df.usage_prev
df['usage_ma3_prev'] = g.usage_hours.transform(lambda s: s.shift(1).rolling(3, min_periods=1).mean())
df['tickets_ma3'] = g.support_tickets.transform(lambda s: s.rolling(3, min_periods=1).mean())
df['usage_ratio'] = (df.usage_hours / df.usage_ma3_prev).replace([np.inf, -np.inf], np.nan)  # prev usage can be 0
df['moy'] = df.snapshot_month.str[5:].astype(int)
df = pd.concat([df, pd.get_dummies(df.plan, prefix='plan', dtype=int)], axis=1)
base = ['seats', 'monthly_fee', 'fee_per_seat', 'tenure_months', 'usage_hours', 'support_tickets',
        'plan_basic', 'plan_pro', 'plan_team']
hist = ['usage_prev', 'usage_chg', 'usage_ma3_prev', 'tickets_ma3', 'usage_ratio']
FEATS = {'snapshot only': base, '+ account history': base + hist,
         '+ history + calendar month': base + hist + ['moy']}
used = sum(FEATS.values(), [])
assert 'lifetime_months' not in used and 'remaining' not in used

## 4. Out-of-time validation
Train on snapshots through 2025-03, skip 2025-04 (its labels overlap the boundary), test on 2025-05 to 2025-11. The test window has an ordinary churn rate and contains **no winter spike**, so this measures normal-period performance only.

In [ ]:
train = df[df.snapshot_month <= '2025-03']; test = df[df.snapshot_month >= '2025-05']
y_tr, y_te = train.churn_next_month, test.churn_next_month
print(len(train), len(test), '| train rate %.3f, test rate %.3f' % (y_tr.mean(), y_te.mean()))
print('accounts in both train and test:', len(set(train.account_ref) & set(test.account_ref)))

def models():
    return {'logreg': make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000)),
            'gbm': HistGradientBoostingClassifier(max_depth=3, learning_rate=0.05, max_iter=200, random_state=0)}

def topk(y, p, k=0.1):
    idx = np.argsort(-p)[:int(len(y) * k)]
    return y.values[idx].mean()

rows = [dict(model='random', features='-', AUC=0.5, PR_AUC=y_te.mean(), top10_precision=y_te.mean())]
for fn, cols in FEATS.items():
    for mn, mdl in models().items():
        Xtr, Xte = train[cols], test[cols]
        if mn == 'logreg':
            Xtr, Xte = Xtr.fillna(Xtr.median()), Xte.fillna(Xtr.median())
        mdl.fit(Xtr, y_tr); p = mdl.predict_proba(Xte)[:, 1]
        rows.append(dict(model=mn, features=fn, AUC=roc_auc_score(y_te, p),
                         PR_AUC=average_precision_score(y_te, p), top10_precision=topk(y_te, p)))
res = pd.DataFrame(rows).round(3); res

In [ ]:
# For contrast ONLY: what the leaky column would 'achieve'
leak_cols = base + ['lifetime_months']
leak = HistGradientBoostingClassifier(max_depth=3, random_state=0).fit(train[leak_cols], y_tr)
print('AUC with leaky lifetime_months: %.3f (not real)' % roc_auc_score(y_te, leak.predict_proba(test[leak_cols])[:, 1]))

## 5. What drives predictions, and is the signal stable?

In [ ]:
from sklearn.inspection import permutation_importance
cols = FEATS['+ account history']
gbm = models()['gbm'].fit(train[cols], y_tr)
pi = permutation_importance(gbm, test[cols], y_te, scoring='roc_auc', n_repeats=10, random_state=0)
pd.Series(pi.importances_mean, cols).sort_values().plot.barh(figsize=(6, 4), title='Permutation importance (test AUC drop)')
plt.show()
t = test.assign(p=gbm.predict_proba(test[cols])[:, 1])
print('AUC by test month:')
print(t.groupby('snapshot_month')[['churn_next_month', 'p']].apply(lambda d: roc_auc_score(d.churn_next_month, d.p)).round(3).to_string())
t['decile'] = pd.qcut(t.p.rank(method='first'), 10, labels=False) + 1
t.groupby('decile').agg(pred=('p', 'mean'), actual=('churn_next_month', 'mean'), n=('p', 'size')).round(3)

## 6. Does it transfer to the winter spike?

In [ ]:
spike = df[df.snapshot_month.between('2024-11', '2025-01')]
norm = df[df.snapshot_month < '2024-10']
mm = models()['gbm'].fit(norm[cols], norm.churn_next_month)
ps = mm.predict_proba(spike[cols])[:, 1]
print('Trained pre-2024-10, tested Nov24-Jan25: AUC %.3f; mean predicted %.3f vs actual %.3f'
      % (roc_auc_score(spike.churn_next_month, ps), ps.mean(), spike.churn_next_month.mean()))

## 7. Conclusions (from my run; re-running should reproduce these)
- **`lifetime_months` is leakage** (AUC 1.0 via `lifetime - tenure`; 0.99 in a model) and was excluded.
- **Honest signal is modest.** Out-of-time (test May-Nov 2025, base rate 9.5%): AUC about 0.68, top-10% of accounts churn at about 24-25% (about 2.5x base rate). Per-month AUC ranges 0.61-0.73.
- **Account history and calendar month did not help**; logistic regression matches gradient boosting, so the signal is simple and weak.
- **Calibration does not transfer**: predicted rates (about 20% on average) are well above the actual 9.5% because training included the winter spike. Ranking works; raw probabilities need recalibration.
- **The Nov 2024 - Jan 2025 spike (about 40-55% churn) is not predictable from these features** (AUC 0.64; predicted 7% vs 47% actual). It looks like a period-level shock we have no data on, and with under two years of data we can't say if it recurs.
- Useful for ranking accounts for retention outreach; not reliable for forecasting the number of churners.